# MAPPO Proximity Source Curriculum 50x50 Outer / 30x30 Inner

Train from scratch inside a `50x50` hidden arena while random hub and food placement stay in the interior `30x30` window. The curriculum keeps two macro food sources throughout, starts with clustered source footprints around those two centers, and shrinks the footprint to two single-tile sources without staging hub-source distance or trainer-side distance rewards.

In [ ]:
from pathlib import Path
import os
import sys

# Set these before importing JAX in this kernel.
os.environ.setdefault("XLA_PYTHON_CLIENT_PREALLOCATE", "false")
os.environ.setdefault("XLA_PYTHON_CLIENT_MEM_FRACTION", "0.35")
if "jax" in sys.modules:
    print("Restart the kernel before rerunning training; JAX was already imported.")

PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "pyproject.toml").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src" / "ant_byte_env").exists():
    raise RuntimeError("Launch this notebook from the cool-antz repo or a subdirectory.")
os.chdir(PROJECT_ROOT)

SRC_ROOT = PROJECT_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

from ant_byte_env import notebook_workflows as workflows

runtime_status = workflows.configure_jax_notebook_runtime()
workflows.assert_notebook_resources_available(runtime_status)
runtime_status


In [ ]:
import importlib

import jax

from ant_byte_env.training.jax_mappo import runner as jax_runner

workflows = importlib.reload(workflows)
jax_runner = importlib.reload(jax_runner)
print(f"JAX device: {jax.devices()[0]}")


## Quick Smoke Run

Run one tiny job before starting the scratch curriculum.

In [ ]:
smoke_metrics = workflows.run_jax_smoke(jax_runner.main)
smoke_metrics


## Curriculum Settings

Edit `experiments/exploration_to_forage_proximity_sources_50x50.json` for durable footprint, reward-scale, margin, or budget changes.

In [ ]:
EXPERIMENT_CONFIG = PROJECT_ROOT / "experiments" / "exploration_to_forage_proximity_sources_50x50.json"
experiment = workflows.load_jax_experiment(EXPERIMENT_CONFIG)
experiment_args = dict(experiment.args)

RUN_DIR = PROJECT_ROOT / "runs" / "notebooks" / experiment.name
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
MEDIA_DIR = RUN_DIR / "media"
ROLLOUT_TILE_SIZE = workflows.NOTEBOOK_ROLLOUT_TILE_SIZE
ROLLOUT_POLICY_TEMPERATURE = workflows.notebook_rollout_policy_temperature(experiment.metadata)
WANDB_VIDEO_MAX_FRAMES = int(experiment.metadata["wandb_video_max_frames"])
WANDB_VIDEO_STAGE_NAMES = tuple(experiment.metadata["wandb_preview_stage_names"])
WANDB_VIDEO_ROLLOUT_COUNT = int(experiment.metadata.get("wandb_preview_rollout_count", 1))
STAGE_UPDATE_MULTIPLIER = float(experiment.metadata.get("stage_update_multiplier", 1.0))
SOURCE_CHECKPOINT = None
if experiment_args.get("load_model"):
    SOURCE_CHECKPOINT = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["load_model"])
    if not SOURCE_CHECKPOINT.exists():
        raise FileNotFoundError(f"Run or restore the source checkpoint first: {SOURCE_CHECKPOINT}")
    experiment_args["load_model"] = str(SOURCE_CHECKPOINT)
BEST_CHECKPOINT_PATH = workflows.resolve_project_path(PROJECT_ROOT, experiment_args["save_best_model"])
experiment_args["save_best_model"] = str(BEST_CHECKPOINT_PATH)

SOURCE_COUNTS = tuple(int(count) for count in experiment.metadata["food_source_counts"])
CLUSTER_RADII = tuple(int(radius) for radius in experiment.metadata["food_cluster_radii"])
CURRICULUM_STAGES = workflows.build_food_cluster_curriculum_stages(
    experiment_args,
    source_counts=SOURCE_COUNTS,
    cluster_radii=CLUSTER_RADII,
    visit_reward_schedule=experiment.metadata.get("visit_reward_schedule"),
    view_reward_schedule=experiment.metadata.get("view_reward_schedule"),
    stage_update_multiplier=STAGE_UPDATE_MULTIPLIER,
)
GLOBAL_UPDATE_CAP = max(int(stage["global_update_cap"]) for stage in CURRICULUM_STAGES)
UPDATE_TIMESTEPS = workflows.update_timesteps(
    num_envs=int(experiment_args["num_envs"]),
    num_steps=int(experiment_args["num_steps"]),
)
WANDB_PROJECT = "cool-antz"
WANDB_ENTITY = None
WANDB_GROUP = experiment.name
WANDB_RUN_NAME = WANDB_GROUP
WANDB_MODE = "online"
CRITIC_TAG = f"{experiment_args.get('critic_architecture', 'mlp').replace('_', '-')}-critic"
COMMON_ARGS = workflows.config_common_args(
    experiment_args,
    exclude=workflows.EXPLORATION_TO_FORAGE_ARG_EXCLUDES,
)
{
    "source_checkpoint": SOURCE_CHECKPOINT,
    "best_checkpoint": BEST_CHECKPOINT_PATH,
    "layout_margin": experiment_args.get("layout_margin"),
    "hub_center_window_size": experiment_args.get("hub_center_window_size"),
    "macro_food_sources": experiment_args.get("food_cluster_count"),
    "distance_bonus": experiment_args.get("distance_bonus"),
    "carrying_hub_distance_bonus": experiment_args.get("carrying_hub_distance_bonus"),
    "critic_architecture": experiment_args.get("critic_architecture"),
    "write_bits": experiment_args.get("write_bits"),
    "per_ant_write_channels": experiment_args.get("per_ant_write_channels"),
    "stage_training_profiles": [
        (
            stage["name"],
            stage["food_sources"],
            stage["food_cluster_radius"],
            stage["food_count"],
            stage["global_update_cap"],
            stage["num_steps"],
            stage["gamma"],
        )
        for stage in CURRICULUM_STAGES
    ],
    "total_updates_per_stage": GLOBAL_UPDATE_CAP,
    "update_timesteps": UPDATE_TIMESTEPS,
}


## Train Proximity Curriculum

In [ ]:
# training_result = workflows.run_forage_curriculum(
#     stages=CURRICULUM_STAGES,
#     checkpoint_dir=CHECKPOINT_DIR,
#     common_args=COMMON_ARGS,
#     update_timesteps_per_stage=UPDATE_TIMESTEPS,
#     global_update_cap=GLOBAL_UPDATE_CAP,
#     train_main=jax_runner.main,
#     initial_checkpoint=SOURCE_CHECKPOINT,
#     wandb_project=WANDB_PROJECT,
#     wandb_entity=WANDB_ENTITY,
#     wandb_group=WANDB_GROUP,
#     wandb_run_name=WANDB_RUN_NAME,
#     wandb_mode=WANDB_MODE,
#     wandb_tags=["exploration-to-forage", "padded-arena", "scratch", "proximity-source-curriculum", "no-distance-aid", "positive-only-reward", "moving-writes", CRITIC_TAG, "50x50", "30x30-inner"],
#     wandb_notes=experiment.metadata["notes"],
#     wandb_artifact_paths=[EXPERIMENT_CONFIG],
#     wandb_artifact_prefix="exploration-to-forage-proximity-sources",
#     checkpoint_name_prefix="jax_mappo_exploration_to_forage_proximity_sources",
#     wandb_video_key_prefix="videos/exploration_to_forage/proximity_sources",
#     wandb_video_max_frames=WANDB_VIDEO_MAX_FRAMES,
#     wandb_video_stage_names=WANDB_VIDEO_STAGE_NAMES,
#     wandb_video_policy_temperature=ROLLOUT_POLICY_TEMPERATURE,
#     wandb_video_rollout_count=WANDB_VIDEO_ROLLOUT_COUNT,
#     checkpoint_video_interval_updates=5000,
#     checkpoint_video_max_frames=WANDB_VIDEO_MAX_FRAMES,
#     checkpoint_video_policy_temperature=ROLLOUT_POLICY_TEMPERATURE,
#     checkpoint_video_rollout_count=WANDB_VIDEO_ROLLOUT_COUNT,
#     checkpoint_video_wandb_key_prefix="videos/exploration_to_forage/proximity_sources/checkpoints",
# )
# FINAL_CHECKPOINT_PATH = training_result["final_checkpoint_path"]
# ROLLOUT_CHECKPOINT_PATH = FINAL_CHECKPOINT_PATH
# training_result


## Optional Local Render and Vault

In [ ]:
import pickle

ROLLOUT_MAZE_SIZE = 41
ROLLOUT_MAZE_CORRIDOR_WIDTH = 5
ROLLOUT_MAZE_CORRIDOR_RADIUS = ROLLOUT_MAZE_CORRIDOR_WIDTH // 2
ROLLOUT_MAZE_POCKET_SIZE = 5
ROLLOUT_NUM_ANTS = int(experiment_args["num_ants"]) * 2
ROLLOUT_MAX_FRAMES = 5000
ROLLOUT_MAZE_ORIGIN = (
    (experiment_args["width"] - ROLLOUT_MAZE_SIZE) // 2,
    (experiment_args["height"] - ROLLOUT_MAZE_SIZE) // 2,
)

def rollout_maze_cell(dx, dy):
    return (ROLLOUT_MAZE_ORIGIN[0] + dx, ROLLOUT_MAZE_ORIGIN[1] + dy)

def rollout_line(start, end):
    start_x, start_y = start
    end_x, end_y = end
    if start_x == end_x:
        step = 1 if end_y >= start_y else -1
        return [(start_x, y) for y in range(start_y, end_y + step, step)]
    if start_y == end_y:
        step = 1 if end_x >= start_x else -1
        return [(x, start_y) for x in range(start_x, end_x + step, step)]
    raise ValueError("maze segments must be horizontal or vertical")

def rollout_expand_cells(cells, radius):
    return list(
        dict.fromkeys(
            (x + dx, y + dy)
            for x, y in cells
            for dy in range(-radius, radius + 1)
            for dx in range(-radius, radius + 1)
            if 0 <= x + dx < ROLLOUT_MAZE_SIZE and 0 <= y + dy < ROLLOUT_MAZE_SIZE
        )
    )

def rollout_rect(x, y, width, height):
    return [(x + dx, y + dy) for dy in range(height) for dx in range(width)]

ROLLOUT_MAZE_WAYPOINTS_REL = [(4, 4), (36, 4), (36, 16)]
ROLLOUT_MAZE_CENTERLINE_REL = list(
    dict.fromkeys(
        cell
        for start, end in zip(ROLLOUT_MAZE_WAYPOINTS_REL, ROLLOUT_MAZE_WAYPOINTS_REL[1:])
        for cell in rollout_line(start, end)
    )
)
ROLLOUT_MAZE_PATH_REL = rollout_expand_cells(
    ROLLOUT_MAZE_CENTERLINE_REL,
    ROLLOUT_MAZE_CORRIDOR_RADIUS,
)
ROLLOUT_MAZE_SIDE_POCKET_REL = list(
    dict.fromkeys(
        [
            *rollout_rect(9, 7, ROLLOUT_MAZE_POCKET_SIZE, ROLLOUT_MAZE_POCKET_SIZE),
            *rollout_rect(17, 7, ROLLOUT_MAZE_POCKET_SIZE, ROLLOUT_MAZE_POCKET_SIZE),
            *rollout_rect(25, 7, ROLLOUT_MAZE_POCKET_SIZE, ROLLOUT_MAZE_POCKET_SIZE),
            *rollout_rect(29, 8, ROLLOUT_MAZE_POCKET_SIZE, ROLLOUT_MAZE_POCKET_SIZE),
            *rollout_rect(29, 12, ROLLOUT_MAZE_POCKET_SIZE, ROLLOUT_MAZE_POCKET_SIZE),
        ]
    )
)
ROLLOUT_MAZE_PATH_SET = {rollout_maze_cell(dx, dy) for dx, dy in ROLLOUT_MAZE_PATH_REL}
ROLLOUT_MAZE_OPEN_POSITIONS = list(
    dict.fromkeys(
        rollout_maze_cell(dx, dy)
        for dx, dy in [*ROLLOUT_MAZE_PATH_REL, *ROLLOUT_MAZE_SIDE_POCKET_REL]
    )
)
ROLLOUT_MAZE_OPEN_SET = set(ROLLOUT_MAZE_OPEN_POSITIONS)
ROLLOUT_MAZE_OBSTACLES = [
    [(x, y) not in ROLLOUT_MAZE_OPEN_SET for x in range(experiment_args["width"])]
    for y in range(experiment_args["height"])
]

ROLLOUT_HUB_POS = rollout_maze_cell(*ROLLOUT_MAZE_WAYPOINTS_REL[0])
ROLLOUT_NORMAL_FOOD_POSITIONS = [rollout_maze_cell(*ROLLOUT_MAZE_WAYPOINTS_REL[-1])]
ROLLOUT_LETHAL_FOOD_POSITIONS = []
if any(rollout_pos in ROLLOUT_MAZE_PATH_SET for rollout_pos in ROLLOUT_LETHAL_FOOD_POSITIONS):
    raise ValueError("lethal rollout positions must stay in side pockets, not on the main path")
for rollout_pos in [ROLLOUT_HUB_POS, *ROLLOUT_NORMAL_FOOD_POSITIONS, *ROLLOUT_LETHAL_FOOD_POSITIONS]:
    if rollout_pos not in ROLLOUT_MAZE_OPEN_SET:
        raise ValueError(f"rollout position {rollout_pos} is not open in the maze")
ROLLOUT_RESET_OPTIONS = {
    "hub_pos": ROLLOUT_HUB_POS,
    "food_positions": ROLLOUT_NORMAL_FOOD_POSITIONS,
    "obstacles": ROLLOUT_MAZE_OBSTACLES,
}
ROLLOUT_RENDER_CHECKPOINT_PATH = MEDIA_DIR / (
    f"{BEST_CHECKPOINT_PATH.stem}_rollout_{ROLLOUT_NUM_ANTS}ants_{ROLLOUT_MAX_FRAMES}frames.pkl"
)
MEDIA_DIR.mkdir(parents=True, exist_ok=True)
with BEST_CHECKPOINT_PATH.open("rb") as checkpoint_file:
    rollout_checkpoint = pickle.load(checkpoint_file)
raw_rollout_args = rollout_checkpoint["args"]
source_rollout_args = dict(raw_rollout_args if isinstance(raw_rollout_args, dict) else vars(raw_rollout_args))
rollout_args = dict(source_rollout_args)
rollout_args["num_ants"] = ROLLOUT_NUM_ANTS
rollout_args["max_steps"] = max(int(rollout_args.get("max_steps", 0)), ROLLOUT_MAX_FRAMES - 1)
rollout_args["food_termination"] = False
rollout_args["lethal_food_count"] = 0
rollout_args["lethal_food_sources"] = 0
rollout_checkpoint["transfer_source_args"] = source_rollout_args
rollout_checkpoint["args"] = rollout_args
with ROLLOUT_RENDER_CHECKPOINT_PATH.open("wb") as checkpoint_file:
    pickle.dump(rollout_checkpoint, checkpoint_file)

rollout_result = workflows.render_jax_checkpoint_rollout(
    run_dir=RUN_DIR,
    checkpoint_path=ROLLOUT_RENDER_CHECKPOINT_PATH,
    media_dir=MEDIA_DIR,
    rollout_filename="jax_mappo_exploration_to_forage_proximity_sources_41x41_5wide_l_maze_20ants_no_lethal_long_rollout.mp4",
    title="JAX MAPPO proximity source curriculum rollout",
    description="Long checkpoint rollout in a 41x41 L-shaped maze with 5-wide corridors, doubled ants, no lethal cookies, and one normal source after the first turn.",
    metadata={
        "experiment_config": str(EXPERIMENT_CONFIG),
        "source_checkpoint": str(SOURCE_CHECKPOINT),
        "best_checkpoint": str(BEST_CHECKPOINT_PATH),
        "rollout_checkpoint": str(ROLLOUT_RENDER_CHECKPOINT_PATH),
        "layout_margin": experiment_args.get("layout_margin"),
        "hub_center_window_size": experiment_args.get("hub_center_window_size"),
        "macro_food_sources": experiment_args.get("food_cluster_count"),
        "critic_architecture": experiment_args.get("critic_architecture"),
        "rollout_source_num_ants": experiment_args.get("num_ants"),
        "rollout_num_ants": ROLLOUT_NUM_ANTS,
        "rollout_max_frames": ROLLOUT_MAX_FRAMES,
        "rollout_maze_size": ROLLOUT_MAZE_SIZE,
        "rollout_maze_corridor_width": ROLLOUT_MAZE_CORRIDOR_WIDTH,
        "rollout_maze_pocket_size": ROLLOUT_MAZE_POCKET_SIZE,
        "rollout_maze_origin": ROLLOUT_MAZE_ORIGIN,
        "rollout_maze_open_positions": ROLLOUT_MAZE_OPEN_POSITIONS,
        "rollout_maze_side_pockets": [
            rollout_maze_cell(dx, dy) for dx, dy in ROLLOUT_MAZE_SIDE_POCKET_REL
        ],
        "rollout_hub_pos": ROLLOUT_HUB_POS,
        "rollout_normal_food_positions": ROLLOUT_NORMAL_FOOD_POSITIONS,
        "rollout_lethal_food_positions": ROLLOUT_LETHAL_FOOD_POSITIONS,
        "food_source_counts": [stage["food_sources"] for stage in CURRICULUM_STAGES],
        "food_cluster_radii": [stage["food_cluster_radius"] for stage in CURRICULUM_STAGES],
    },
    max_frames=ROLLOUT_MAX_FRAMES,
    tile_size=ROLLOUT_TILE_SIZE,
    policy_temperature=ROLLOUT_POLICY_TEMPERATURE,
    reset_options=ROLLOUT_RESET_OPTIONS,
    reuse_existing=False,
    wandb_project=WANDB_PROJECT,
    wandb_entity=WANDB_ENTITY,
    wandb_group=WANDB_GROUP,
    wandb_run_name=f"{WANDB_GROUP}_rollout",
    wandb_mode="disabled",
    wandb_video_key=None,
    # wandb_step=training_result["stage_metrics"][-1].get("curriculum_global_step"),
)
rollout_result
